# Eligibility-aware RAG credit-card recommender: final end-to-end notebook
**Runtime:** Colab T4 GPU. **Needs:** `Thesis-Coding-final.zip` in `MyDrive/Thesis/`.

Run top to bottom. Steps 1-9 take about 15 minutes and validate everything (integrity checks, 33 unit/end-to-end tests, real-model sanity check). Steps 11-14 are the long evaluation (several hours on a T4). **Every generation is cached on Drive and every result is saved to Drive**, so after a disconnect re-run Steps 1-5 and 8, then simply re-run the evaluation cells: finished work is replayed instantly.

## Step 1: Mount Drive and check the GPU

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
free, total = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name(0)} | free {free/1024**3:.1f} / {total/1024**3:.1f} GiB")

Mounted at /content/drive
GPU: Tesla T4 | free 14.5 / 14.6 GiB


## Step 2: Settings (run before any project import)

In [2]:
import os

ZIP_PATH = "/content/drive/MyDrive/Thesis/Thesis-Coding-final.zip"
RESULTS_DIR = "/content/drive/MyDrive/Thesis/results_final"        # all numbers + predictions + RESULTS.md
os.makedirs(RESULTS_DIR, exist_ok=True)

os.environ["USE_STUB_LLM"] = "false"
os.environ["LLM_MODEL_NAME"] = "Qwen/Qwen2.5-7B-Instruct"             # 4-bit; fallback: "Qwen/Qwen2.5-3B-Instruct"
os.environ["LLM_CACHE_PATH"] = "/content/drive/MyDrive/Thesis/llm_cache.jsonl"   # survives disconnects
os.environ["LLM_CACHE"] = "1"
os.environ["LLM_BATCH_SIZE"] = "8"                                    # per-card prompts are generated 8 at a time
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # optional

## Step 3: Unpack the project

In [3]:
!rm -rf /content/Thesis-Coding && mkdir -p /content/Thesis-Coding
!unzip -q -o "$ZIP_PATH" -d /content/Thesis-Coding
%cd /content/Thesis-Coding
assert "round 4" in open("CHANGES.md").read().lower(), "Old zip: upload Thesis-Coding-final.zip"
!ls

/content/Thesis-Coding
CHANGES.md  data		 pyproject.toml    src
config	    evaluation		 README.md	   tests
dashboard   EVALUATION_NOTES.md  requirements.txt


## Step 4: Install dependencies

In [4]:
%pip install -q -r requirements.txt "requests==2.32.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.0/258.0 kB 28.9 MB/s eta 0:00:00


## Step 5: Build the FAISS index and chunk table

In [5]:
!python -m src.pipeline
assert os.path.exists("artifacts/faiss.index") and os.path.exists("artifacts/chunks.parquet")

modules.json: 100% 349/349 [00:00<00:00, 1.47MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 504kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 25.5MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 328kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.96MB/s]

model.safetensors: downloading bytes:  90% 82.2M/90.9M [00:01<00:00, 75.4MB/s, 6.46MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 47.2MB/s, 7.92MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 50.4MB/s, 8.68MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 6508.87it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.75MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 55.2MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 110MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 739kB/s]
config.json: 100% 190/190 [00:00<00:00, 989kB/s]
Wrote artifacts/faiss.index and artifacts/chunks.parquet


## Step 6: Integrity checks and tests
`verify_setup` confirms the rule documents agree with the eligibility table, the gold labels equal the oracle, the index matches the documents, and the grounded/ungrounded prompts are built correctly. **Both must pass before any result is trusted.** Expect `33 passed`.

In [6]:
!python -m evaluation.verify_setup
!python -m pytest -q tests 2>&1 | tail -3

PASS  rule documents agree with the eligibility table
PASS  gold labels equal the oracle
PASS  index/chunk artefacts match the rule documents
PASS  grounded/ungrounded prompts are built correctly

All integrity checks passed.
.................................                                        [100%]
33 passed in 99.35s (0:01:39)


## Step 7: Plumbing check with the stub LLM (about 1 minute)
Stub numbers only test the code path; they are not model results.

In [7]:
!python -m evaluation.run_ablation --use-stub-llm --split test --limit 6 --out-dir /content/stub_results --arms A2 A3 A5 A6 2>&1 | grep -E "===|Results saved|Error|Traceback"

=== A1: grounded vs ungrounded (B1) ===
=== Research Question summary ===
=== A2: top-k sweep (rules kept per card) ===
=== A3: prompt pattern on/off ===
=== A5: chunking granularity ===
=== A6: normal vs reworded profiles ===
Results saved to /content/stub_results/results.json


## Step 8: Load the real model and sanity-test single and batched generation

In [8]:
from src.infrastructure.llm_client import LocalLLMClient

client = LocalLLMClient(os.environ["LLM_MODEL_NAME"])
print("model:", client.model_name, "| max_new_tokens:", client.max_new_tokens)
print(client.generate("Reply with exactly: GPU test passed."))
print(client.generate_batch(["Reply with exactly: A", "Reply with exactly: B"]))

model: Qwen/Qwen2.5-7B-Instruct | max_new_tokens: 1536


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

GPU test passed.
['A', 'B']


## Step 9: Real-pipeline check on 3 profiles (grounded and ungrounded)
**Pass:** grounded `STRUCTURE OK` (8 parsed, nothing missing/extra) **and** the ungrounded arm also returns all 8 cards (otherwise its errors would be output-format failures, not wrong knowledge). `WRONG` is model accuracy.

In [9]:
from pathlib import Path
from evaluation.run_ablation import load_gold_set, build_retriever, make_llm_clients
from src.model import RAGRecommender

rows = Path("data/gold_test_set.jsonl").read_text(encoding="utf-8").splitlines()
Path("/content/gold_smoke.jsonl").write_text("\n".join(rows[:3]) + "\n", encoding="utf-8")
records = load_gold_set("/content/gold_smoke.jsonl")

grounded_llm, ungrounded_llm = make_llm_clients(use_stub=False)
retriever = build_retriever(4)
grounded = RAGRecommender(retriever, grounded_llm)
ungrounded = RAGRecommender(retriever, ungrounded_llm)

structural_ok = True
for i, rec in enumerate(records):
    expected = {(d.bank, d.card): d.recommended for d in rec.expected}
    for name, system in (("GROUNDED  ", grounded), ("UNGROUNDED", ungrounded)):
        decisions, k = system.recommend(rec.profile)
        actual = {(d.bank, d.card): d.recommended for d in decisions}
        missing, extra = set(expected) - set(actual), set(actual) - set(expected)
        wrong = {c[1]: (expected[c], actual[c]) for c in expected.keys() & actual.keys() if expected[c] != actual[c]}
        if name.strip() == "GROUNDED":
            structural_ok &= (not missing and not extra and len(decisions) == len(expected))
        print(f"Profile {i} {name}: parsed={len(decisions)}/{len(expected)} MISSING={len(missing)} WRONG={wrong or '-'}")
    d0 = grounded.recommend(rec.profile)[0][0]
    print("   e.g.", d0.card, d0.recommended, "|", d0.reason[:150], "|", d0.cited_chunk_id)
assert structural_ok, "Grounded output structure problem: paste this output"
print("\nSTRUCTURE OK")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Profile 0 GROUNDED  : parsed=8/8 MISSING=0 WRONG=-
Profile 0 UNGROUNDED: parsed=7/8 MISSING=1 WRONG={'HDFC MoneyBack+': (True, False)}
   e.g. HDFC MoneyBack+ True | The applicant meets the minimum annual income requirement of ₹2,50,000 and has a CIBIL score of 801, which is above the required 650. The applicant is | hdfc_cards#0
Profile 1 GROUNDED  : parsed=8/8 MISSING=0 WRONG=-
Profile 1 UNGROUNDED: parsed=2/8 MISSING=6 WRONG={'HDFC Regalia': (False, True)}
   e.g. HDFC MoneyBack+ True | The applicant meets the age requirement (18-65) and has a CIBIL score of 768, which is above the minimum of 650. The annual income of ₹7,31,330 is abo | hdfc_cards#0
Profile 2 GROUNDED  : parsed=8/8 MISSING=0 WRONG={'HDFC MoneyBack+': (True, False)}
Profile 2 UNGROUNDED: parsed=6/8 MISSING=2 WRONG=-
   e.g. HDFC MoneyBack+ False | The applicant meets the age requirement of 18 or above, has a CIBIL score of 650 or above, and is self-employed, which is an allowed employment type.  | hdfc_cards#0

STRUC

## Step 9b: Free GPU/RAM before the evaluation subprocesses

In [10]:
import gc, torch
for n in ("client", "grounded_llm", "ungrounded_llm", "grounded", "ungrounded", "retriever"):
    globals().pop(n, None)
gc.collect(); torch.cuda.empty_cache()
print(f"{torch.cuda.mem_get_info()[0]/1024**3:.1f} GiB free")

9.1 GiB free


## Step 10 (optional): quick pilot, 10 profiles
Gives a first real number and the seconds per profile for planning. Skip if you are short on time.

In [11]:
!python -m evaluation.run_ablation --split test --limit 10 --out-dir /content/pilot --arms 2>&1 | tr '\r' '\n' | grep -v "it/s" | grep -E "===|accuracy|RQ|verdict|returned"

=== A1: grounded vs ungrounded (B1) ===
  grounded_accuracy: 0.3
  ungrounded_accuracy: 0.0
  verdict: HYPOTHESIS NOT SUPPORTED
  grounded_cards_returned_rate: 1.0
  ungrounded_cards_returned_rate: 0.7875
=== Research Question summary ===
RQ1 (gold-set classification vs the eligibility oracle; RAGAS faithfulness is NOT computed here): profile accuracy = 0.300; per-card precision=1.000 recall=0.417 f1=0.588 accuracy=0.912 (majority-class baseline accuracy=0.850; cards returned=1.000)
RQ2 (context precision/recall, proxy = all cards of the requested banks): mean_context_precision=1.000 mean_context_recall=1.000
RQ3 (hallucination reduction via McNemar): p=0.2500; relative error reduction=30.0%; verdict=HYPOTHESIS NOT SUPPORTED
RQ4 (explanation correctness): label_consistency_rate=0.300; citation_validity_rate=0.988 (cited chunk retrieved for the query and matching the decision's bank/card)


## Step 11: MAIN RESULT: test split (40 profiles), A1 + RQ1 / RQ2 / RQ3 / RQ4
The test split is every 3rd gold profile. Results are merged into `results.json` on Drive.

In [12]:
!python -m evaluation.run_ablation --split test --out-dir $RESULTS_DIR --arms 2>&1 | tr '\r' '\n' | grep --line-buffered -v "it/s" | tee $RESULTS_DIR/main_test40.log


Loaded 40 gold profiles (test split) from data/gold_test_set.jsonl
CAVEAT: figures below are an OPTIMISTIC UPPER BOUND computed on a synthetic gold set against curated, representative (not live) documents for exactly two banks (SBI, HDFC); there is no runtime validation gate in the live system -- these are the only faithfulness checks that ever happen.





Loading weights:   0%|          | 1/339 [00:04<24:27,  4.34s/it]

Loading weights:   1%|          | 2/339 [00:08<24:35,  4.38s/it]

Loading weights:   1%|          | 4/339 [00:09<10:12,  1.83s/it]

Loading weights:   1%|▏         | 5/339 [00:09<07:53,  1.42s/it]

Loading weights:   2%|▏         | 6/339 [00:10<06:22,  1.15s/it]




































































































100%|██████████| 40/40 [22:09<00:00, 36.22s/it]
                                               

100%|██████████| 40/40 [08:49<00:00, 14.88s/it]
                                               

=== A1: grounded

## Step 12: Ablation arms (20 test profiles each; `--no-core` keeps the main result intact)
One cell per arm; run them in any order.

In [13]:
!python -m evaluation.run_ablation --split test --limit 20 --no-core --out-dir $RESULTS_DIR --arms A3 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|prompt_pattern" | tee $RESULTS_DIR/a3.log

=== A3: prompt pattern on/off ===
  prompt_pattern_on_accuracy: 0.350
  prompt_pattern_off_accuracy: 0.750


In [14]:
!python -m evaluation.run_ablation --split test --limit 20 --no-core --out-dir $RESULTS_DIR --arms A5 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|accuracy" | tee $RESULTS_DIR/a5.log

=== A5: chunking granularity ===
  fine_grained_1_rule_per_chunk_accuracy: 0.350
  coarse_multi_rule_per_chunk_accuracy: 0.850


In [15]:
!python -m evaluation.run_ablation --split test --limit 20 --no-core --out-dir $RESULTS_DIR --arms A2 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|top_k" | tee $RESULTS_DIR/a2.log

=== A2: top-k sweep (rules kept per card) ===
  top_k=1: accuracy=0.150
  top_k=2: accuracy=0.050
  top_k=3: accuracy=0.350
  top_k=4: accuracy=0.350


In [16]:
!python -m evaluation.run_ablation --split test --limit 40 --no-core --out-dir $RESULTS_DIR --arms A6 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|balanced|normal_|reworded_" | tee $RESULTS_DIR/a6.log

=== A6: normal vs reworded profiles ===
  (balanced sample: 40 profiles drawn from the full gold set)
  normal_accuracy: 0.350
  reworded_accuracy: 0.700
  normal_n: 20
  reworded_n: 20


## Step 13 (optional): A4 model selection (Qwen2.5-3B vs 7B)

In [17]:
!python -m evaluation.run_ablation --split test --limit 20 --no-core --out-dir $RESULTS_DIR --arms A4 --a4-models Qwen/Qwen2.5-3B-Instruct Qwen/Qwen2.5-7B-Instruct 2>&1 | tr '\r' '\n' | grep --line-buffered -E "===|Qwen" | tee $RESULTS_DIR/a4.log

=== A4: model selection ===
  Qwen/Qwen2.5-3B-Instruct: 0.05
  Qwen/Qwen2.5-7B-Instruct: 0.35


## Step 14 (optional but recommended): held-out confirmation on the dev split
The 40 test profiles were looked at during earlier development rounds. Running the final configuration on 40 dev profiles (never evaluated before) gives a clean estimate to report next to the main result.

In [18]:
!python -m evaluation.run_ablation --split dev --limit 40 --out-dir $RESULTS_DIR/dev_confirmation --arms 2>&1 | tr '\r' '\n' | grep --line-buffered -v "it/s" | tee $RESULTS_DIR/dev_confirmation.log


Loaded 40 gold profiles (dev split) from data/gold_test_set.jsonl
CAVEAT: figures below are an OPTIMISTIC UPPER BOUND computed on a synthetic gold set against curated, representative (not live) documents for exactly two banks (SBI, HDFC); there is no runtime validation gate in the live system -- these are the only faithfulness checks that ever happen.





Loading weights:   0%|          | 1/339 [00:04<24:27,  4.34s/it]

Loading weights:   1%|          | 2/339 [00:08<24:34,  4.38s/it]

Loading weights:   1%|          | 4/339 [00:09<10:11,  1.82s/it]

Loading weights:   1%|▏         | 5/339 [00:09<07:53,  1.42s/it]

Loading weights:   2%|▏         | 6/339 [00:10<06:22,  1.15s/it]









































































































100%|██████████| 40/40 [27:42<00:00, 42.01s/it]
                                               

100%|██████████| 40/40 [11:24<00:00, 17.38s/it]
                                               

=== A1: grou

## Step 15: Build the shareable report (`RESULTS.md`, saved on Drive)

In [19]:
!python -m evaluation.make_report --results-dir $RESULTS_DIR

# Results

*LLM:* `Qwen/Qwen2.5-7B-Instruct` | *embedding:* `sentence-transformers/all-MiniLM-L6-v2` | *per-card prompting:* True | *split:* test | *profiles (core):* 40 | *run:* 2026-10-14T12:25:03+00:00

## A1 / RQ3: grounded vs ungrounded

| metric | value |
|---|---|
| profiles (n) | 40 |
| grounded profile accuracy | 0.400 |
| ungrounded profile accuracy | 0.000 |
| profiles fixed by grounding (b) | 16 |
| profiles broken by grounding (c) | 0 |
| McNemar p-value | 0.000 |
| relative error reduction | 0.400 |
| grounded cards returned | 1.000 |
| ungrounded cards returned | 0.719 |
| meets 0.90 stretch target | False |
| verdict | HYPOTHESIS SUPPORTED |

> Warning: the ungrounded arm returned fewer than 95% of the cards; part of its error may be output-format failure, not wrong knowledge.

## RQ1: gold-set classification (RAGAS faithfulness not computed)

| metric | value |
|---|---|
| precision | 0.953 |
| recall | 0.554 |
| f1 | 0.701 |
| accuracy | 0.891 |
| majority_class_accur

In [20]:
# dev confirmation report (if you ran Step 14)
import os
if os.path.exists(f"{RESULTS_DIR}/dev_confirmation/results.json"):
    get_ipython().system(f"python -m evaluation.make_report --results-dir {RESULTS_DIR}/dev_confirmation")

# Results

*LLM:* `Qwen/Qwen2.5-7B-Instruct` | *embedding:* `sentence-transformers/all-MiniLM-L6-v2` | *per-card prompting:* True | *split:* dev | *profiles (core):* 40 | *run:* 2026-10-14T14:30:47+00:00

## A1 / RQ3: grounded vs ungrounded

| metric | value |
|---|---|
| profiles (n) | 40 |
| grounded profile accuracy | 0.225 |
| ungrounded profile accuracy | 0.000 |
| profiles fixed by grounding (b) | 9 |
| profiles broken by grounding (c) | 0 |
| McNemar p-value | 0.004 |
| relative error reduction | 0.225 |
| grounded cards returned | 1.000 |
| ungrounded cards returned | 0.675 |
| meets 0.90 stretch target | False |
| verdict | HYPOTHESIS SUPPORTED |

> Warning: the ungrounded arm returned fewer than 95% of the cards; part of its error may be output-format failure, not wrong knowledge.

## RQ1: gold-set classification (RAGAS faithfulness not computed)

| metric | value |
|---|---|
| precision | 0.794 |
| recall | 0.355 |
| f1 | 0.491 |
| accuracy | 0.825 |
| majority_class_accurac

## What to send your supervisor
`MyDrive/Thesis/results_final/`: `RESULTS.md` (all tables and error analysis), `results.json` (every number with model, split and timings), `predictions_A1_*.jsonl` (per-profile expected vs actual with the model's reasons), the `*.log` files, and the project zip with `EVALUATION_NOTES.md` and `CHANGES.md`.

## Step 16 (optional): serve the API
Run in its own session: `!uvicorn src.api:app --host 0.0.0.0 --port 8000`. With the real LLM each request issues 8 batched per-card generations.